In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES']='0'

import torch
import torch.nn.functional as F
import numpy as np
import argparse
from scipy import misc
import cv2

from lib.networks import PVT_CASCADE
from utils.dataloader import test_dataset
import matplotlib.pyplot as plt

/nfs/P111yhchen/anaconda3/envs/medseg/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/P111yhchen/code/medseg/CASCADE/lib/pvtv2.py:388: UserWarning: Overwriting pvt_v2_b0 in registry with lib.pvtv2.pvt_v2_b0. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  class pvt_v2_b0(PyramidVisionTransformerImpr):
/home/P111yhchen/code/medseg/CASCADE/lib/pvtv2.py:398: UserWarning: Overwriting pvt_v2_b1 in registry with lib.pvtv2.pvt_v2_b1. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  class pvt_v2_b1(PyramidVisionTransformerImpr):
/home/P111yhchen/code/medseg/CASCADE/lib/pvtv2.py:406: UserWarning: Overwriting pvt_v2_b2 in registry with lib.pvtv2.pvt_v2_b2. This is because the n

In [2]:
testsize = 352
#device_id = 7
#device = torch.device(f'cuda:{device_id}')
#torch.cuda.set_device(device_id)

dataset = 'polyp'
pth_root = '/nfs/P111yhchen/code/medseg/CASCADE/models/{}/emcad/'.format(dataset)
#pth_root = '/home/P111yhchen/code/medseg/CASCADE/models/polyp/emcad-no_lgag'
pth_path = os.path.join(pth_root, 'best.pth')


model = PVT_CASCADE(n_class=1).cuda()
model.eval()
#model.load_state_dict(torch.load(pth_path, map_location="cuda:0"), strict=False)

from thop import profile
in0 = torch.randn(1, 3, 352, 352).cuda()
in4 = torch.randn(1, 512, testsize//32, testsize//32).cuda()
in3 = torch.randn(1, 320, testsize//16, testsize//16).cuda()
in2 = torch.randn(1, 128, testsize//8, testsize//8).cuda()
in1 = torch.randn(1, 64, testsize//4, testsize//4).cuda()
macs, params = profile(model.decoder, inputs=(in4, [in3, in2, in1]))
print(params, macs)

pvt_v2_b2


/home/P111yhchen/code/medseg/CASCADE/lib/networks.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  save_model = torch.load(path)


Model decoder:  created, param count: 327586
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU6'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_upsample() for <class 'torch.nn.modules.upsampling.Upsample'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.AdaptiveMaxPool2d'>.
327586.0 690970496.0


In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).cuda()
STD = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).cuda()

In [ ]:
#sab = list(model.decoder.sab.conv.parameters())
#for t in sab:
#    print(t.shape)
#    t = torch.abs(t)
#    print(t[:,0].min(), t[:,0].max())
#    print(t[:,1].min(), t[:,1].max())
#    fig, ax = plt.subplots(1, 2)
#    ax[0].imshow(t[:,0].squeeze().detach().cpu().numpy())
#    ax[1].imshow(t[:,1].squeeze().detach().cpu().numpy())
#    plt.show()

In [ ]:
for _data_name in ['CVC-ClinicDB', 'Kvasir','CVC-300',  'CVC-ColonDB', 'ETIS-LaribPolypDB']:
#for _data_name in ['ETIS-LaribPolypDB']:
#for _data_name in ['TrainDataset']:
    ##### put data_path here #####
    data_path = '/home/P111yhchen/data/polyp/TestDataset/{}'.format(_data_name)
    #data_path = '/local_data/dataset/polyp/{}'.format(_data_name)
    print('Evaluating ' + data_path)
    
    image_root = '{}/images/'.format(data_path)
    gt_root = '{}/masks/'.format(data_path)
    num1 = len(os.listdir(gt_root))
    test_loader = test_dataset(image_root, gt_root, 352)
    DSC = 0.0
    JACARD = 0.0
    preds = []
    gts = []
    for i in range(num1):
        image, gt, name = test_loader.load_data()
        gt = np.asarray(gt, np.float32)
        gt /= (gt.max() + 1e-8)
        image = image.cuda()
        img = torch.clamp(image * STD + MEAN, 0, 1)
        h,w = gt.shape
        gt_tensor = torch.tensor(gt).cuda().view(1,1,h,w)
        gt_tensor = F.upsample(gt_tensor, size=(352,352), mode='bilinear', align_corners=False)       
        #print(name)
        results = model(image) # forward
        pred_cls = results[-1].sigmoid()
        #loss_ce, loss_dice = structure_loss(res4, gt_tensor)

        # eval Dice
        res = F.upsample(results[-1], size=gt.shape, mode='bilinear', align_corners=False)       
        pred_msk = 1.*(res>0)
        res = pred_msk.data.cpu().numpy().squeeze() # apply sigmoid aggregation for binary segmentation
        res = (res - res.min()) / (res.max() - res.min() + 1e-8)      
        
        input = res
        target = np.where(np.array(gt) >= 0.5, 1, 0)
        
        preds.append(input)
        gts.append(gt)
        
        N = gt.shape
        smooth = 1
        input_flat = np.reshape(input, (-1))
        target_flat = np.reshape(target, (-1))
        intersection = (input_flat * target_flat)
        union = input_flat + target_flat - intersection
        
        jacard = ((np.sum(intersection)+smooth)/(np.sum(union)+smooth))
        jacard = '{:.4f}'.format(jacard)
        jacard = float(jacard)
        JACARD += jacard
        
        dice = (2 * intersection.sum() + smooth) / (input.sum() + target.sum() + smooth)
        dice = '{:.4f}'.format(dice)
        dice = float(dice)
        DSC += dice
        #print(dice)
        img_show = img.squeeze().permute(1,2,0).detach().cpu().numpy()
        msk_show = torch.cat([gt_tensor, gt_tensor, gt_tensor])
        msk_show = msk_show.squeeze().permute(1,2,0).detach().cpu().numpy()
        #cls_show = torch.cat([gt_tensor, pred_cls, pred_cls])
        #cls_show = cls_show.squeeze().permute(1,2,0).detach().cpu().numpy()
        pred_show = torch.cat([pred_cls, pred_cls, pred_cls])
        pred_show = pred_show.squeeze().permute(1,2,0).detach().cpu().numpy()
        #loss_show = loss_ce.squeeze().detach().cpu().numpy()

        save_path = os.path.join(pth_root, _data_name)
        #cv2.imwrite(os.path.join(save_path, name), 255*pred_show)
        #
        #fig, ax = plt.subplots(1, 3)
        #plt.subplots_adjust(wspace=0.05)
        #ax[0].imshow(img_show)
        #ax[0].axis('off')
        #ax[1].imshow(msk_show)
        #ax[1].axis('off')
        #ax[2].imshow(pred_show)
        #ax[2].axis('off')
        #plt.show()
    
    print('*****************************************************')
    print('Dice Score: ' + str(DSC/num1))
    print('Jacard Score: ' + str(JACARD/num1))
    
    print(_data_name, 'Finish!')
    print('*****************************************************')